# PI-CAI Z-SSMNet Pipeline — 10-Patient Sanity Test

This notebook runs the full pipeline on **10 randomly selected patients** with **5 training epochs** to verify everything works end-to-end before committing to the full 1500-patient run.

**Architecture:**
- Source data: Read from Google Drive (`PI-CAI_pre-processed/`)
- Processing: Runs on Colab's fast local NVMe SSD
- Checkpoints: Saved back to Google Drive (`PI-CAI_Results/`)

In [ ]:
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define paths
SOURCE_DATA_DIR = "/content/drive/MyDrive/PI-CAI_pre-processed"
RESULTS_FOLDER = "/content/drive/MyDrive/PI-CAI_Results"
WORKSPACE_DIR = "/content/PI-CAI_Workspace/baseline"

# 3. Sanity test configuration
MAX_CASES = "10"     # Only process 10 patients
MAX_EPOCHS = "5"     # Only train for 5 epochs

# Export all environment variables
os.environ["SOURCE_DATA_DIR"] = SOURCE_DATA_DIR
os.environ["RESULTS_FOLDER"] = RESULTS_FOLDER
os.environ["WORKSPACE_DIR"] = WORKSPACE_DIR
os.environ["MAX_CASES"] = MAX_CASES
os.environ["MAX_EPOCHS"] = MAX_EPOCHS

print(f"Source data: {SOURCE_DATA_DIR}")
print(f"Results: {RESULTS_FOLDER}")
print(f"Workspace: {WORKSPACE_DIR}")
print(f"Max cases: {MAX_CASES}")
print(f"Max epochs: {MAX_EPOCHS}")

In [ ]:
# 4. Clone repositories to fast local disk
!mkdir -p /content/PI-CAI_Workspace

import os
if not os.path.exists("/content/PI-CAI_Workspace/baseline"):
    !cd /content/PI-CAI_Workspace && git clone https://github.com/HemishJain09/PI-CAI-Baseline.git baseline
else:
    !cd /content/PI-CAI_Workspace/baseline && git pull

if not os.path.exists("/content/PI-CAI_Workspace/Z-SSMNet"):
    !cd /content/PI-CAI_Workspace && git clone https://github.com/yuanyuan29/Z-SSMNet.git Z-SSMNet

print("Repositories ready.")

In [ ]:
# 5. Install dependencies
import os
os.environ["SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL"] = "True"
!pip install -q -r $WORKSPACE_DIR/requirements.txt
!pip install -q git+https://github.com/DIAGNijmegen/nnUNet.git@1.7.0-3
print("Dependencies installed.")

In [ ]:
# 6. Run the full pipeline (live output)
import os
os.chdir(os.environ["WORKSPACE_DIR"])
os.environ["SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL"] = "True"
!chmod +x run_nnunet_pipeline.sh
!./run_nnunet_pipeline.sh

## What to look for

If the sanity test succeeds, you should see:
1. **STEP 1**: `Subset mode: randomly selected 10 cases for processing.`
2. **STEP 2**: nnUNet planning and preprocessing completes without errors
3. **STEP 2.5**: `Successfully wrote splits_final.pkl with 5 folds.`
4. **STEP 3**: Zonal masks injected successfully
5. **STEP 4**: Training starts and completes 5 epochs with loss decreasing

Once verified, to run the full 1500-patient training:
1. Delete the checkpoint markers: `!rm -f .format_complete .preprocess_complete .zonal_integration_complete`
2. Set `MAX_CASES = ""` and `MAX_EPOCHS = ""` in Cell 1
3. Re-run all cells